# 🧪 vLLM Step2 ドライラン（GPU 不要）

`twolven/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-AWQ-MTP` を vLLM で配信する構成のうち、**推論以外の全工程**を CPU で検証する。

| # | 検証 | 実推論 |
|---|---|---|
| 1 | vLLM install / バージョン / 所要時間 | 不要 |
| 2 | `vllm serve --help` のフラグ有無（起動ラダーの前提） | 不要 |
| 3 | Dataset マウント + symlink ステージング | 不要 |
| 4 | プロキシ + UI + `--enforce-sampling` | モック上流 |
| 5 | cloudflared 越しの UI / models / streaming / OpenAI互換 | モック上流 |
| 6 | Gist / keepalive / 後片付け | 不要 |

> モック上流（8000番）が vLLM の代役。T4 のカーネル挙動だけは確認できない。

**設定**: Accelerator `None` / Internet `On` / Add Data: `yamahei21python/qwen38-27b-uncensored-hauhaucs-awq-mtp`, `my-cloudflared-bin`


---
### 🧩 共通ランタイム（proxy / tunnel / keepalive / verify）

`runtime/` の真実源をビルダーが連結して埋め込んでいます。


In [ ]:
# ============================================================
# 共通ランタイム（runtime/*.py から自動生成。手で編集しない）
# proxy / tunnel / keepalive / verify の配管をここに集約する。
# ============================================================
# ── runtime/net.py ────────────────────────────────────────
"""トンネル URL 取得と HTTP 疎通の共通処理（生成コードへ埋め込む）。"""
import json
import re
import time
import urllib.error
import urllib.request

# hosts は表駆動。URL の出所（ログ解析 / 管理API）と待ち時間だけが違う。
TUNNEL_HOSTS = {
    "cloudflared": {
        "kind": "log",
        "url_re": re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com"),
        "wait_seconds": 90,
    },
    "ngrok": {
        "kind": "api",
        "api_url": "http://127.0.0.1:4040/api/tunnels",
        "wait_seconds": 60,
    },
}


def kc_probe_url(host, log_path=None):
    """現在の公開URLを返す（無ければ ""）。host は cloudflared / ngrok。"""
    spec = TUNNEL_HOSTS[host]
    if spec["kind"] == "api":
        try:
            with urllib.request.urlopen(spec["api_url"], timeout=3) as r:
                d = json.loads(r.read().decode())
            for t in d.get("tunnels", []):
                u = t.get("public_url", "")
                if u.startswith("https://"):
                    return u
        except Exception:
            return ""
        return ""
    if not log_path:
        return ""
    try:
        with open(log_path) as f:
            m = spec["url_re"].search(f.read())
        return m.group(0) if m else ""
    except OSError:
        return ""


def kc_wait_url(host, log_path=None, seconds=None, interval=1, progress_every=10):
    """公開URLが出るまで待つ。seconds 未指定なら host ごとの既定値。"""
    if seconds is None:
        seconds = TUNNEL_HOSTS[host]["wait_seconds"]
    for i in range(seconds):
        url = kc_probe_url(host, log_path)
        if url:
            return url
        if progress_every and i % progress_every == progress_every - 1:
            print(f"⏳ URL 待ち ({i + 1}/{seconds})...")
        time.sleep(interval)
    return ""


def kc_wait_http(url, attempts=20, interval=3, timeout=10, opener=None):
    """200 が返るまで待つ。成功したら経過秒（(i+1)*interval）、失敗なら None。"""
    opener = opener or urllib.request.urlopen
    for i in range(attempts):
        try:
            with opener(url, timeout=timeout) as r:
                if r.status == 200:
                    return (i + 1) * interval
        except Exception:
            pass
        time.sleep(interval)
    return None


def kc_retry(fn, tries=6, delay=5, skip_http_errors=True):
    """一過性の通信エラーを吸収する。HTTP 応答は「答え」なので既定で再試行しない。"""
    last = None
    for i in range(max(1, tries)):
        try:
            return fn()
        except urllib.error.HTTPError as e:
            if skip_http_errors:
                raise
            last = e
        except Exception as e:
            last = e
        if i + 1 < tries:
            wait = delay * (i + 1)
            print(f"  ⚠ 一時的な通信エラー（{i + 1}/{tries}）: {last} "
                  f"→ {wait}s 待って再試行")
            time.sleep(wait)
    raise last


def kc_post_json(url, payload, timeout=60, headers=None, opener=None):
    """JSON POST のレスポンスを返す（ストリームは呼び出し側で読む）。"""
    h = {"Content-Type": "application/json",
         "ngrok-skip-browser-warning": "true"}
    if headers:
        h.update(headers)
    req = urllib.request.Request(
        url, data=json.dumps(payload).encode(), headers=h, method="POST")
    return (opener or urllib.request.urlopen)(req, timeout=timeout)

# ── runtime/proc.py ───────────────────────────────────────
"""プロセス・GPU 状態の確認と後片付け（生成コードへ埋め込む）。"""
import subprocess
import time


def _kc_live(pids, timeout=5):
    """zombie を除いた PID を返す。kill 直後は未 reap の Z が pgrep に残る。"""
    if not pids:
        return []
    try:
        out = subprocess.run(
            ["ps", "-o", "pid=,stat=", "-p", ",".join(pids)],
            capture_output=True, text=True, timeout=timeout,
        ).stdout
    except Exception:
        return pids
    live = []
    for line in out.splitlines():
        parts = line.split()
        if len(parts) >= 2 and not parts[1].startswith("Z"):
            live.append(parts[0])
    return live


def kc_proc_state(name, match="exact", timeout=5):
    """alive / DEAD / unknown。match=exact は pgrep -x、argv は pgrep -f。"""
    try:
        flag = "-x" if match == "exact" else "-f"
        out = subprocess.run(["pgrep", flag, name], capture_output=True,
                             text=True, timeout=timeout).stdout
        pids = [p for p in out.split() if p.strip()]
        return "alive" if _kc_live(pids, timeout) else "DEAD"
    except Exception:
        return "unknown"


def kc_gpu_usage(timeout=5):
    """nvidia-smi の使用率/VRAM を 1 行で返す（CPU 環境では N/A）。"""
    try:
        return subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=timeout,
        ).stdout.strip().replace("\n", " | ")
    except Exception:
        return "N/A"


def _kc_argv(pattern):
    """"-x ngrok" → ["-x", "ngrok"]（shell を介さない）。"""
    return pattern.split()


def kc_pids(pattern, timeout=15):
    """pgrep の引数文字列（例: "-x ngrok"）に一致する PID を返す。"""
    try:
        r = subprocess.run(["pgrep", *_kc_argv(pattern)],
                           capture_output=True, text=True, timeout=timeout)
        return _kc_live([p for p in r.stdout.split() if p.strip()], timeout)
    except Exception:
        return []


def kc_stop_processes(targets, term_wait=3, final_wait=2):
    """(pgrep_pattern, label) を TERM → 待機 → KILL で止め、残存ラベルを返す。"""
    for pattern, label in targets:
        if kc_pids(pattern):
            subprocess.run(["pkill", "-TERM", *_kc_argv(pattern)],
                           capture_output=True)
            print(f"  TERM → {label}")
    time.sleep(term_wait)
    for pattern, label in targets:
        if kc_pids(pattern):
            subprocess.run(["pkill", "-KILL", *_kc_argv(pattern)],
                           capture_output=True)
            print(f"  KILL → {label}（残存していた）")
    time.sleep(final_wait)
    return [label for pattern, label in targets if kc_pids(pattern)]

# ── runtime/storage.py ────────────────────────────────────
"""検証成果物の永続化と失敗記録（生成コードへ埋め込む）。"""
import os

# `kaggle kernels output` が zip 化するのは /kaggle/working だけ。
# /output は公式の永続先ではないが、書ければ保険になるので両方へ書く。
KC_OUT_DIRS = ("/kaggle/working", "/output")


def kc_save_output(name, text, dirs=KC_OUT_DIRS):
    """1 つでも書ければ True（成功条件に使う）。"""
    saved = []
    for d in dirs:
        try:
            os.makedirs(d, exist_ok=True)
            with open(os.path.join(d, name), "w") as f:
                f.write(text)
            saved.append(d)
        except Exception as e:
            print(f"  ⚠ {d} 書き出し失敗 ({e})")
    if saved:
        print(f"  📄 {name} 保存済: {', '.join(saved)}")
        return True
    print(f"  ⚠ 成果物を1つも保存できませんでした: {name}")
    return False


def kc_fail(msg, tail_path=None, dirs=KC_OUT_DIRS):
    """失敗原因を永続領域に残してから raise（ログ API は当てにできない）。"""
    tail = ""
    try:
        if tail_path and os.path.isfile(tail_path):
            tail = open(tail_path, errors="replace").read()[-3000:]
    except Exception:
        pass
    kc_save_output("FAILED.txt", f"{msg}\n\n{tail}", dirs)
    raise RuntimeError(msg)

# ── runtime/keepalive.py ──────────────────────────────────
"""キープアライブ設定ファイルと /stop の共通処理（生成コードへ埋め込む）。"""
import json
import os
import urllib.request


def kc_read_conf(path, default_minutes=60, max_minutes=480,
                 on_invalid="fail", fail=None, quiet_clamp=False):
    """(run, minutes, mtime) を返す。RUN=0 は (0, default_minutes, mtime)。

    on_invalid: "fail" は fail(msg)（未指定なら ValueError）、
                "zero" は停止扱い（RUN=0 と同じ）。
    quiet_clamp: proxy が書く既定値を上限で頭打ちにするとき黙る。
    """
    try:
        with open(path) as f:
            d = dict(l.strip().split("=", 1) for l in f
                     if l.strip() and "=" in l)
        run = int(d.get("RUN", 1))
        minutes = int(d.get("MINUTES", default_minutes))
    except OSError:
        run, minutes = 1, default_minutes
    except ValueError:
        run, minutes = 1, default_minutes
    try:
        mtime = os.path.getmtime(path)
    except OSError:
        mtime = None
    if run <= 0:
        return 0, default_minutes, mtime
    if minutes <= 0:
        msg = f"KEEPALIVE_MINUTES={minutes} は無効（1〜{max_minutes} で指定）"
        if on_invalid == "zero":
            return 0, default_minutes, mtime
        if fail:
            fail(msg)
        raise ValueError(msg)
    if minutes > max_minutes:
        if not (quiet_clamp and minutes == default_minutes):
            print(f"⚠ KEEPALIVE_MINUTES={minutes} → {max_minutes} に制限")
        minutes = max_minutes
    return run, minutes, mtime


def kc_write_conf(path, run, minutes=None):
    with open(path, "w") as f:
        f.write(f"RUN={run}\n")
        if minutes is not None:
            f.write(f"MINUTES={minutes}\n")


def kc_conf_mtime(path):
    """keepalive.conf の更新時刻。proxy がチャット送信ごとに書き直す。"""
    try:
        return os.path.getmtime(path)
    except OSError:
        return 0.0


def kc_stop_via_proxy(port, auth=None, timeout=30):
    headers = {"X-Kc-Stop": "1"}
    if auth:
        headers["Authorization"] = auth
    req = urllib.request.Request(
        f"http://127.0.0.1:{port}/stop", data=b"", method="POST",
        headers=headers)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, json.loads(r.read())

# ── runtime/proxy.py ──────────────────────────────────────
"""auth_proxy の起動と認証なし検査（生成コードへ埋め込む）。"""
import subprocess
import time
import urllib.error
import urllib.request


def kc_start_proxy(port, upstream, chat_html, proxy_path,
                   extra_args=(), env=None, log_path="/tmp/auth_proxy.log",
                   wait_seconds=30, label="プロキシ", fail=None):
    """auth_proxy.py を起動し /healthz を待つ。失敗時は fail() か RuntimeError。"""
    subprocess.run(["pkill", "-f", "auth_proxy.py"], capture_output=True)
    time.sleep(1)
    cmd = ["python3", proxy_path, "--port", str(port),
           "--upstream", upstream, "--chat-html", chat_html, *extra_args]
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, env=env, stdout=log, stderr=log)
    print(f"🔄 proxy PID: {proc.pid}")

    elapsed = None
    for i in range(wait_seconds):
        try:
            with urllib.request.urlopen(
                    f"http://127.0.0.1:{port}/healthz", timeout=3) as r:
                if r.status == 200:
                    elapsed = i + 1
                    break
        except Exception:
            pass
        time.sleep(1)
    if elapsed is None:
        try:
            print(open(log_path).read()[-2000:])
        except OSError:
            pass
        msg = f"{label}の起動に失敗しました（{log_path}）"
        if fail:
            fail(msg, log_path)
        raise RuntimeError(msg)
    print(f"✅ {label}起動完了（{elapsed}s）")
    return proc


def kc_assert_no_auth(port, path="/api/tags", fail=None, timeout=10):
    """401/403 なら認証が有効 = 設定ミス。上流未起動の 5xx は正常。"""
    req = urllib.request.Request(f"http://127.0.0.1:{port}{path}")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            print(f"  ✅ 認証なし → HTTP {r.status}")
            return True
    except urllib.error.HTTPError as e:
        if e.code in (401, 403):
            msg = f"認証なしで HTTP {e.code} — 認証が有効になっています"
            if fail:
                fail(msg)
            raise RuntimeError(msg)
        print(f"  ✅ 認証なし → HTTP {e.code}（401 でない）")
        return True
    except Exception as e:
        msg = f"プロキシ検査に失敗: {type(e).__name__}: {e}"
        if fail:
            fail(msg)
        raise



In [ ]:
# 設定（GPU 不要）
import glob as _glob
import json
import os
import re
import subprocess
import sys
import time

SERVED_NAME = "qwen3.8-27b-awq"
VLLM_PORT = 8000
PROXY_PORT = 11435
MTP_ENABLED = True
MTP_METHOD = "mtp"
MTP_TOKENS = 2
MAXLEN_CANDIDATES = [32768, 16384, 8192]
TP_SIZE = 2

PROXY_SRC_B64 = "..."
CHAT_SRC_B64 = "..."
PROXY_PATH = "/usr/local/bin/auth_proxy.py"
CHAT_PATH = "/usr/local/bin/llama-chat.html"

OUT_DIRS = ("/kaggle/working", "/output")


def _find_input(slug):
    cands = []
    for pat in ("/kaggle/input/" + slug,
                "/kaggle/input/*/" + slug,
                "/kaggle/input/*/*/" + slug):
        cands += [p for p in _glob.glob(pat) if os.path.isdir(p)]
    for p in sorted(cands):
        try:
            if os.listdir(p):
                return p
        except OSError:
            continue
    return cands[0] if cands else "/kaggle/input/" + slug


MODEL_SRC = _find_input("qwen38-27b-uncensored-hauhaucs-awq-mtp")


save_output = kc_save_output
fail = kc_fail


_files = sorted(os.listdir(MODEL_SRC)) if os.path.isdir(MODEL_SRC) else []
print("モデル:", MODEL_SRC)
print("直下ファイル:", len(_files))
for _f in _files:
    _p = os.path.join(MODEL_SRC, _f)
    if os.path.isfile(_p):
        print(f"  {_f:34s} {os.path.getsize(_p):>15,} B")
if "config.json" not in _files:
    fail("config.json がありません。Add Data を確認してください。")
if "model.safetensors" not in _files:
    fail("model.safetensors がありません。Dataset を確認してください。")


---
### 1️⃣ vLLM install とフラグ判定（所要時間を実測）


In [ ]:
# vLLM を入れて、起動ラダーが前提にしているフラグの有無を確認する
import subprocess
import sys
import time

t0 = time.time()
_r = subprocess.run([sys.executable, "-c", "import vllm"],
                    capture_output=True, text=True)
install_sec = 0.0
if _r.returncode == 0:
    print("ℹ vLLM は既に入っている（install 時間 0）")
else:
    print("⬇️  pip install vllm 開始")
    _p = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-input", "vllm"],
        capture_output=True, text=True, timeout=3600)
    install_sec = time.time() - t0
    print((_p.stderr or "")[-1500:])
    if _p.returncode != 0:
        fail("vLLM install 失敗（上のログ）")

# torchaudio は transformers が import する。vLLM の pip install で torch だけが
# 差し替わると CUDA 版数が不一致になり、vllm CLI が即死する
# （実測: vllm 0.32.0 / torch 2.13.0+cu130 に対し torchaudio 2.8.3+cu128）。
#
# 検出は「実際に落ちる経路」で行う。`import vllm.entrypoints.cli.main` だけでは
# 不十分だった（v2 実測）: benchmark.main 等は main() の中で遅延 import される
# ため、深い依存（transformers → torchaudio）の不整合を見逃す。
# torchaudio の整合は serve より前に単独で確認する（v2 の真因）。
# PyPI の torchaudio 2.11.0 は +cu130 ビルド（wheel の version.py で実測）なので、
# torch 2.13.0+cu130 とそのまま一致する。
def _ta_check():
    r = subprocess.run([sys.executable, "-c", "import torchaudio"],
                       capture_output=True, text=True, timeout=300)
    return r.returncode == 0, ((r.stdout or "") + (r.stderr or ""))[-900:]


_ta_good, _ta_err = _ta_check()
if not _ta_good:
    print("⚠ torchaudio 不整合 → 2.11.0(+cu130) を --no-deps で入れ直す")
    print(_ta_err)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    "--no-deps", "--force-reinstall", "torchaudio==2.11.0"],
                   capture_output=True, text=True, timeout=1800)
    _ta_good, _ta_err = _ta_check()
if not _ta_good:
    # vLLM が torchaudio を使うのは音声モデルだけ（midashenglm/audio.py 等）。
    # このモデルは音声不要なので、外すのが最終手段として妥当。
    print("⚠ まだ不整合 → torchaudio を削除")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                    "torchaudio"], capture_output=True, text=True, timeout=600)
    _ta_good, _ta_err = _ta_check()
print(f"torchaudio import: {'OK' if _ta_good else 'NG'}")

_PREFLIGHT_CMD = ["vllm", "serve", "--help"]


def _cli_ok():
    r = subprocess.run(_PREFLIGHT_CMD, capture_output=True, text=True,
                       timeout=300)
    out = (r.stdout or "") + (r.stderr or "")
    bad = ("Traceback" in out) or ("RuntimeError" in out) or len(out) < 500
    return (r.returncode == 0 and not bad), out[-1500:]


_ok, _err = _cli_ok()
if not _ok:
    # transformers 側は ImportError を握るので、外すのが最短（通信不要）。
    # vLLM 0.32 は torchaudio==2.11.0 を要求する（METADATA 実測）。
    # 版を揃えて入れ直すのが本筋。駄目なら外す（transformers は ImportError を握る）。
    print("⚠ vLLM CLI が起動しない → torchaudio を 2.11.0 で入れ直す")
    print(_err)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    "--no-deps", "--force-reinstall", "torchaudio==2.11.0"],
                   capture_output=True, text=True, timeout=1800)
    _ok, _err = _cli_ok()
if not _ok:
    print("⚠ まだ失敗 → torchaudio を外して再試行")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                    "torchaudio"], capture_output=True, text=True, timeout=600)
    _ok, _err = _cli_ok()
if not _ok:
    print("⚠ まだ失敗 → torchaudio を torch に合わせて入れ直す")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    "-U", "torchaudio"], capture_output=True, text=True,
                   timeout=3600)
    _ok, _err = _cli_ok()
if not _ok:
    # help が壊れていても serve が壊れているとは限らない（実測: CPU では
    # transformers ではなく arg_utils 側で別の失敗をする）。ここで止めず、
    # 起動を試す。壊れていれば最初の試行が即座に落ちるだけで済む。
    print("⚠ vLLM CLI の確認に失敗 → そのまま起動を試す")
    print(_err)

import torch
import vllm
print(f"✅ vLLM {vllm.__version__} / torch {torch.__version__} "
      f"/ CUDA {torch.version.cuda}")
print(f"⏱  install 所要: {install_sec/60:.1f} 分")

_h = subprocess.run(["vllm", "serve", "--help"],
                    capture_output=True, text=True, timeout=180)
HELP = (_h.stdout or "") + (_h.stderr or "")
# 判定が外れたときの唯一の手掛かり。ログと成果物の両方に残す。
print(f"vllm serve --help: exit={_h.returncode} {len(HELP)} 文字")
print(HELP[:1500])
save_output("vllm_help.txt", HELP[:20000])
HELP_OK = (_h.returncode == 0 and len(HELP) > 500
           and "Traceback (most recent call last)" not in HELP)

# ここに列挙した flag だけを avail に入れる。avail[flag] を直接引く箇所が
# あるため、後から使う flag を足し忘れると HELP 失敗時（= CPU では常に）に
# KeyError で即死する（v6 で実測）。
NEEDED = ["--speculative-config", "--trust-remote-code", "--reasoning-parser",
          "--enable-auto-tool-choice", "--tool-call-parser",
          "--disable-log-requests", "--kv-cache-dtype",
          "--gpu-memory-utilization", "--max-num-seqs",
          "--served-model-name", "--tensor-parallel-size",
          "--language-model-only"]
avail = {f: (f in HELP) for f in NEEDED}
if not HELP_OK:
    # 悲観側に倒す。0.32 で廃止されたフラグを付けると未知引数で即死する。
    avail = {f: False for f in NEEDED}
for f, ok in avail.items():
    print(("✅" if ok else "❌"), f)

missing = [f for f in ("--tensor-parallel-size", "--served-model-name",
                       "--max-num-seqs", "--gpu-memory-utilization")
           if not avail[f]]
if not HELP_OK:
    print("⚠ ヘルプ本文が取れない → フラグ判定は不能。"
          "本番は悲観側（任意フラグを落とす）で動く")
elif missing:
    # ここで止めると「help の書式が変わっただけ」でも全損になる。
    # 記録して先へ進む（本番ビルダーは同じ状況でも起動を試す）。
    print(f"⚠ 必須フラグが見つかりません（書式変更の可能性）: {missing}")

# 起動ラダーを組み立てる（実起動はしない）
# 量子化は指定しない（config の quantization_config は pack-quantized。
# awq 固定は誤り。詳細 notes/vllm-cpu-findings.md）
CORE = ["vllm", "serve", "<stage>", "--served-model-name", SERVED_NAME,
        "--host", "127.0.0.1", "--port", str(VLLM_PORT),
        "--tensor-parallel-size", str(TP_SIZE), "--dtype", "half",
        "--max-num-seqs", "2",
        "--gpu-memory-utilization", "0.90", "--kv-cache-dtype", "auto"]
# 本番と同じくテキスト専用で起動（視覚塔は落とす）。
if avail["--language-model-only"]:
    CORE.append("--language-model-only")
OPT = []
if avail["--trust-remote-code"]:
    OPT.append("--trust-remote-code")
if avail["--disable-log-requests"]:   # 0.32 では廃止（--enable-log-requests）
    OPT.append("--disable-log-requests")
if avail["--reasoning-parser"]:
    OPT += ["--reasoning-parser", "qwen3"]
if avail["--enable-auto-tool-choice"] and avail["--tool-call-parser"]:
    OPT += ["--enable-auto-tool-choice", "--tool-call-parser", "qwen3_xml"]

SPEC = None
if MTP_ENABLED and avail["--speculative-config"]:
    SPEC = {"method": MTP_METHOD, "num_speculative_tokens": MTP_TOKENS}

PLAN = []
if SPEC:
    PLAN.append(("mtp", SPEC, MAXLEN_CANDIDATES[0], True))
    PLAN.append(("mtp", SPEC, MAXLEN_CANDIDATES[1], True))
PLAN.append(("plain", None, MAXLEN_CANDIDATES[0], False))
PLAN.append(("plain", None, MAXLEN_CANDIDATES[1], False))
PLAN.append(("plain", None, 8192, False))
PLAN.append(("triton", None, 8192, False))   # 0.32 に V0 は無い（明示 backend）

print()
print("起動ラダー（GPU 実行時にこの順で試す）:")
for i, (lab, sp, ml, use_opt) in enumerate(PLAN, 1):
    print(f"  {i}. spec={lab} max_len={ml} opt_flags={'on' if use_opt else 'off'}")
save_output("dryrun_plan.txt", json.dumps(
    {"vllm": vllm.__version__, "torch": torch.__version__,
     "install_sec": install_sec, "flags": avail,
     "plan": [[l, s, m, o] for l, s, m, o in PLAN]},
    indent=2, ensure_ascii=False) + "\n")


---
### 2️⃣ Dataset のマウントと symlink ステージング

MTP の draft head を出し入れしても実体は壊れないことを確認。


In [ ]:
# symlink ステージング（実体コピーなし = disk を食わない）
import os
import shutil

STAGE = "/kaggle/tmp/vllm-model"
MTP_FILE = "model-mtp.safetensors"


def _stage_model(with_mtp):
    if os.path.isdir(STAGE):
        shutil.rmtree(STAGE)          # symlink のみなので実体は消えない
    os.makedirs(STAGE, exist_ok=True)
    n = 0
    for fn in sorted(os.listdir(MODEL_SRC)):
        if fn.startswith("."):
            continue
        if fn == MTP_FILE and not with_mtp:
            continue
        src = os.path.join(MODEL_SRC, fn)
        if not os.path.isfile(src):
            continue
        os.symlink(src, os.path.join(STAGE, fn))
        n += 1
    return n


n_on = _stage_model(True)
on_files = sorted(os.listdir(STAGE))
n_off = _stage_model(False)
off_files = sorted(os.listdir(STAGE))
print(f"MTP ON : {n_on} files  mtp={'model-mtp.safetensors' in on_files}")
print(f"MTP OFF: {n_off} files  mtp={'model-mtp.safetensors' in off_files}")
print(f"ステージ合計サイズ（実体参照）: "
      f"{sum(os.path.getsize(os.path.join(STAGE, f)) for f in off_files)/1e9:.1f} GB")
print(f"Dataset 実体は無傷: {os.path.isfile(os.path.join(MODEL_SRC, 'model.safetensors'))}")

if "model-mtp.safetensors" in on_files and \
        "model-mtp.safetensors" not in off_files and n_off >= 3:
    print("✅ ステージング OK")
else:
    fail(f"ステージング異常: on={n_on} off={n_off}")


---
### 3️⃣ モック vLLM 上流（8000番）

本物の代役。`/v1/models`, `/v1/chat/completions`（stream 対応）、受信ボディの記録。


In [ ]:
# モック上流: vLLM の OpenAI 互換 API を最小再現する
import json
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

LAST = {}


class Mock(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *a):
        pass

    def _json(self, obj, code=200):
        b = json.dumps(obj, ensure_ascii=False).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(b)))
        self.end_headers()
        self.wfile.write(b)

    def do_GET(self):
        p = self.path.split("?")[0]
        if p == "/v1/models":
            self._json({"object": "list", "data": [
                {"id": SERVED_NAME, "object": "model",
                 "max_model_len": 32768, "owned_by": "mock"}]})
        elif p == "/__last":
            self._json(LAST)
        else:
            self._json({"error": "not found"}, 404)

    def do_POST(self):
        global LAST
        n = int(self.headers.get("Content-Length") or 0)
        raw = self.rfile.read(n) if n else b"{}"
        try:
            body = json.loads(raw)
        except Exception:
            body = {}
        LAST = {"path": self.path, "body": body}

        if body.get("stream"):
            self.send_response(200)
            self.send_header("Content-Type", "text/event-stream")
            self.send_header("Cache-Control", "no-cache")
            self.send_header("Transfer-Encoding", "chunked")
            self.end_headers()
            for i in range(12):
                chunk = {"id": "mock", "object": "chat.completion.chunk",
                         "choices": [{"index": 0, "delta":
                                      {"content": f"tok{i} "}}]}
                payload = ("data: " + json.dumps(chunk) + "\n\n").encode()
                self.wfile.write(b"%x\r\n" % len(payload) + payload + b"\r\n")
                self.wfile.flush()
                time.sleep(0.15)
            done = b"data: [DONE]\n\n"
            self.wfile.write(b"%x\r\n" % len(done) + done + b"\r\n")
            self.wfile.write(b"0\r\n\r\n")
            self.wfile.flush()
            return

        self._json({
            "id": "mock", "object": "chat.completion",
            "model": body.get("model", SERVED_NAME),
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {"role": "assistant",
                                     "content": "OK. (mock)"}}],
            "usage": {"prompt_tokens": 8, "completion_tokens": 4,
                      "total_tokens": 12},
        })


subprocess.run(["pkill", "-f", "mock_vllm"], capture_output=True)
_server = ThreadingHTTPServer(("127.0.0.1", VLLM_PORT), Mock)
threading.Thread(target=_server.serve_forever, daemon=True).start()
time.sleep(0.5)
import urllib.request
with urllib.request.urlopen(f"http://127.0.0.1:{VLLM_PORT}/v1/models",
                            timeout=5) as r:
    _d = json.loads(r.read())
print(f"✅ モック起動: {VLLM_PORT} models={[m['id'] for m in _d['data']]}")


---
### 4️⃣ プロキシ起動（`--enforce-sampling`）


In [ ]:
# プロキシ + UI を配置して起動（本番と同じ引数）
import base64
import os

with open(PROXY_PATH, "wb") as f:
    f.write(base64.b64decode(PROXY_SRC_B64))
with open(CHAT_PATH, "wb") as f:
    f.write(base64.b64decode(CHAT_SRC_B64))
print(f"auth_proxy.py {os.path.getsize(PROXY_PATH):,} B / "
      f"llama-chat.html {os.path.getsize(CHAT_PATH):,} B")

proxy_proc = kc_start_proxy(
    PROXY_PORT, f"http://127.0.0.1:{VLLM_PORT}", CHAT_PATH, PROXY_PATH,
    extra_args=("--enforce-sampling",), fail=fail)


---
### 5️⃣ cloudflared quick tunnel


In [ ]:
# cloudflared 起動（Dataset 優先 / 無ければ DL）
import os
import shutil
import subprocess
import time
import urllib.request

CF_BIN = "/usr/local/bin/cloudflared"


def _open_public(url, timeout=15):
    """公開URLを開く。失敗時は IPv4 優先で再解決して 1 回だけ再試行。

    Kaggle の VM は AAAA を返すのに IPv6 経路が無いことがあり、
    urlopen が "Network is unreachable" になる（dryrun v6 で実測）。
    """
    import socket
    try:
        return urllib.request.urlopen(url, timeout=timeout)
    except Exception as first:
        err = first
    orig = socket.getaddrinfo

    def _v4(host, port, family=0, type=0, proto=0, flags=0):
        return orig(host, port, socket.AF_INET, type, proto, flags)

    socket.getaddrinfo = _v4
    try:
        return urllib.request.urlopen(url, timeout=timeout)
    except Exception:
        raise err
    finally:
        socket.getaddrinfo = orig


NONE = _find_input("my-cloudflared-bin")
src = None
for cand in (os.path.join(NONE, "cloudflared"),
             os.path.join(NONE, "bin", "cloudflared")):
    if os.path.isfile(cand) and os.path.getsize(cand) > 1024 * 1024:
        src = cand
        break
if src:
    shutil.copy2(src, CF_BIN)
    subprocess.run(["chmod", "+x", CF_BIN])
    print(f"✅ cloudflared 復元: {src}")
elif not os.path.isfile(CF_BIN):
    subprocess.run(
        "curl -fsSL -o /usr/local/bin/cloudflared "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared",
        shell=True, check=True, timeout=300)

TUNNEL_NAME = "cloudflared"
TUNNEL_LOG = "/tmp/cloudflared.log"
RESTART_CMD = [CF_BIN, "tunnel", "--url", f"http://127.0.0.1:{PROXY_PORT}",
               "--no-autoupdate"]

subprocess.run(["pkill", "-x", "cloudflared"], capture_output=True)
time.sleep(1)
with open(TUNNEL_LOG, "w") as log:
    subprocess.Popen(RESTART_CMD, stdout=log, stderr=log)

public_url = kc_wait_url("cloudflared", TUNNEL_LOG)
if not public_url:
    print(open(TUNNEL_LOG).read()[-3000:])
    fail("公開URL取得失敗", TUNNEL_LOG)

print(f"🌍 {public_url}")
with open("/tmp/public_url.txt", "w") as f:
    f.write(public_url)
save_output("public_url.txt", public_url + "\n")

elapsed = kc_wait_http(f"{public_url}/healthz", attempts=20, interval=3,
                       opener=_open_public)
if elapsed is None:
    print("⚠ 公開URLへの疎通が 60s 以内に取れませんでした（次の検証で再試行）")
else:
    print(f"✅ トンネル確立（{elapsed}s）")


---
### 6️⃣ 公開URL経由の検証（モック上流）


In [ ]:
# 5点検証（実推論以外は本番と同一経路）
import json
import time as _time
import urllib.error
import urllib.request

with open("/tmp/public_url.txt") as f:
    public_url = f.read().strip()

lines = [f"PUBLIC_URL={public_url}", f"MODEL={SERVED_NAME}", ""]
failures = []


def _judge(idx, label, ok, detail):
    print(("✅" if ok else "❌"), f"{label}: {detail}")
    lines.append(f"[{idx}] {label} -> {'PASS' if ok else 'FAIL'}: {detail}")
    if not ok:
        failures.append(f"[{idx}] {label}: {detail}")


def _post(url, payload, timeout=120, stream=False):
    return kc_post_json(url, payload, timeout=timeout)


def _with_retry(fn, tries=3, delay=3):
    """公開 URL は cloudflared 側の一時障害で `Network is unreachable` になる。
    実測: v5 で UI と /v1/models だけ落ち、localhost 系は全部通った。
    一過性なので数回だけ粘る（本番で 20 分消費した後に誤 fail しないため）。
    """
    return kc_retry(fn, tries=tries, delay=delay, skip_http_errors=False)


CHAT = {"model": SERVED_NAME,
        "messages": [{"role": "user", "content": "Say OK."}],
        "max_tokens": 32}

# [1] UI
try:
    with _with_retry(lambda: _open_public(
            f"{public_url}/", timeout=60)) as r:
        html = r.read()
        ok_ui = (r.status == 200 and len(html) > 10000
                 and html.lstrip()[:15].lower().startswith(b"<!doctype html"))
        _judge(1, "ui", ok_ui,
               f"HTTP {r.status} {len(html)}B")
except Exception as e:
    _judge(1, "ui", False, f"{type(e).__name__}: {e}")

# [2] /v1/models
try:
    with _with_retry(lambda: _open_public(
            f"{public_url}/v1/models", timeout=60)) as r:
        md = json.loads(r.read())
    ids = [m.get("id") for m in md.get("data", [])]
    _judge(2, "models", r.status == 200 and SERVED_NAME in ids,
           f"ids={ids}")
except Exception as e:
    _judge(2, "models", False, f"{type(e).__name__}: {e}")

# [3] streaming
try:
    resp = _post(f"{public_url}/v1/chat/completions",
                 dict(CHAT, stream=True), timeout=120)
    times = []
    t0 = _time.time()
    with resp as r:
        for raw in r:
            if not raw.strip():
                continue
            times.append(_time.time() - t0)
    total = _time.time() - t0
    if len(times) >= 3:
        first, spread = times[0], times[-1] - times[0]
        ok = spread > 0.2 and first < total * 0.8
        detail = (f"chunks={len(times)} first={first:.2f}s "
                  f"total={total:.2f}s spread={spread:.2f}s")
    else:
        ok, detail = False, f"chunks={len(times)}"
    _judge(3, "streaming", ok, detail)
except Exception as e:
    _judge(3, "streaming", False, f"{type(e).__name__}: {e}")

# [4] OpenAI 互換（temperature=0 でも 200 + 生成）
try:
    with _post(f"{public_url}/v1/chat/completions",
               dict(CHAT, temperature=0, repetition_penalty=1.0)) as r:
        st = r.status
        cj = json.loads(r.read())
    text = "".join((c.get("message") or {}).get("content") or ""
                   for c in cj.get("choices", []))
    usage = cj.get("usage") or {}
    _judge(4, "openai-compat", st == 200 and bool(text or usage),
           f"HTTP {st} reply={text[:30]!r} usage={usage}")
except Exception as e:
    _judge(4, "openai-compat", False, f"{type(e).__name__}: {e}")

# [5] enforce-sampling が上流まで届いているか（モックに記録されたボディ）
try:
    with urllib.request.urlopen(
            f"http://127.0.0.1:{VLLM_PORT}/__last", timeout=10) as r:
        last = json.loads(r.read())
    b = last.get("body") or {}
    temp = b.get("temperature")
    rp = b.get("repetition_penalty")
    ok = temp is not None and float(temp) >= 0.7 and \
        rp is not None and float(rp) >= 1.05
    _judge(5, "enforce-sampling", ok, f"upstream temp={temp} rp={rp}")
except Exception as e:
    _judge(5, "enforce-sampling", False, f"{type(e).__name__}: {e}")

# [6] 会話（履歴が上流までそのまま渡る）
# 実推論は GPU でしか試せないが、proxy が messages を欠落・改変しないことは
# ここで確認できる（role 順・日本語・system の保持）。
try:
    conv = [
        {"role": "system", "content": "あなたは日本語で簡潔に答えます。"},
        {"role": "user", "content": "私の名前はコウヘイです。"},
        {"role": "assistant", "content": "コウヘイさん、覚えました。"},
        {"role": "user", "content": "私の名前は？"},
    ]
    resp = _post(f"{public_url}/v1/chat/completions",
                 {"model": SERVED_NAME, "messages": conv, "max_tokens": 32},
                 timeout=120)
    with resp as r:
        r.read()
    with _open_public(f"http://127.0.0.1:{VLLM_PORT}/__last", timeout=10) as r:
        sent = ((json.loads(r.read()).get("body") or {}).get("messages") or [])
    ok6 = sent == conv
    _judge(6, "conversation-history", ok6,
           f"messages={len(sent)} equal={ok6}")
except Exception as e:
    _judge(6, "conversation-history", False, f"{type(e).__name__}: {e}")

report = "\n".join(lines)
print()
print("=" * 60)
print(report)
print("=" * 60)
if not save_output("verify_result.txt", report + "\n"):
    failures.append("verify_result.txt 保存失敗")
if failures:
    fail("検証失敗: " + "; ".join(failures))
print("✅ 6/6 パス（モック上流）— 推論以外は本番同一")


---
### 7️⃣ キープアライブ（2分・CPU）


In [ ]:
# keepalive 2分（本番は 480 分）。プロキシ/トンネルの死活だけ見る
import time
import urllib.request

KEEPALIVE_MINUTES = 8
KEEPALIVE_CONF = "/tmp/keepalive.conf"


def _proc(name):
    return kc_proc_state(name, "exact" if name == "cloudflared" else "argv")


def _read_conf():
    run, minutes, _mtime = kc_read_conf(KEEPALIVE_CONF, KEEPALIVE_MINUTES,
                                        KEEPALIVE_MINUTES, on_invalid="zero")
    return run, minutes


print(f"Public URL: {public_url}")
log = []
for i in range(1, KEEPALIVE_MINUTES + 1):
    run, minutes = _read_conf()
    if run <= 0:
        print(f"RUN=0 で {i}周目に終了")
        break
    time.sleep(60)
    p_st, t_st = _proc("auth_proxy.py"), _proc("cloudflared")
    line = (f"[{time.strftime('%H:%M:%S')}] proxy={p_st} tunnel={t_st}")
    if i % 1 == 0:
        try:
            with urllib.request.urlopen(f"{public_url}/healthz", timeout=15) as r:
                line += " | tunnel=OK" if r.status == 200 else f" | HTTP{r.status}"
        except Exception as e:
            line += f" | tunnel=ERR({type(e).__name__})"
    log.append(line)
    print(line)

save_output("keepalive.txt", "\n".join(log) + "\n")

# 後片付け
left = kc_stop_processes([("-x cloudflared", "cloudflared"),
                          ("-f auth_proxy.py", "auth_proxy")])
if "_server" in globals():
    _server.shutdown()
print("🧹 停止済（proxy / cloudflared / mock）")
if left:
    print(f"⚠ 残存: {left}")
